In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,Activation,Flatten,Conv2D,MaxPooling2D
import pickle

training_path=
test_path=
x=pickle.load(open("x.pickle","rb"))
y=pickle.load(open("y.pickle","rb"))
X=x/255.0


Pré-traitement des données

In [ ]:
# Supposons un fichier CSV avec colonnes : lat, lon, wind_speed
data = pd.read_csv("donnees_meteo.csv")

# Définir la résolution de la grille souhaitée (ex : 0.1°)
grid_lat = np.linspace(data["lat"].min(), data["lat"].max(), num=100)
grid_lon = np.linspace(data["lon"].min(), data["lon"].max(), num=100)
grid_lon, grid_lat = np.meshgrid(grid_lon, grid_lat)

# Interpolation des données brutes vers la grille régulière
grid_wind = griddata(
    (data["lon"], data["lat"]),
    data["wind_speed"],
    (grid_lon, grid_lat),
    method="linear"  # ou "nearest"/"cubic"
)

# Visualisation
plt.imshow(grid_wind, cmap="viridis")
plt.colorbar(label="Vitesse du vent (m/s)")
plt.show()

# Normalisation entre 0 et 1
scaler = MinMaxScaler()
grid_wind_normalized = scaler.fit_transform(grid_wind)

# Formatage pour TensorFlow (batch_size, height, width, channels)
# Ajout d'une dimension pour le canal (ex : 1 canal pour le vent)
input_data = grid_wind_normalized.reshape((1, *grid_wind.shape, 1))

print("Shape final pour le CNN :", input_data.shape)
# Output : (1, 100, 100, 1)

# Sauvegarder les données
np.save("wind_grid.npy", input_data)

## Modèle

In [ ]:
model=Sequential()
model.add(Conv2D(64,(3,3),input_shape=X.shape[1:]))
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2,2)))


model.add(Conv2D(64,(3,3),input_shape=X.shape[1:]))
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2,2)))


model.add(Conv2D(64,(3,3),input_shape=X.shape[1:]))
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Flatten())
model.add(Dense(64))

model.add(Dense(1))
model.add(Activation('sigmoid'))
model.compile(loss="binary_crossentropy",
              optimizer="adam",
              metrics=['accuracy'])
model.fit(x,y,batch_size=32,epochs=10,validation_split=0.1)